# ETL da camada Silver para camada Gold - Microsoft Security Incident Prediction

Este notebook realiza o ETL (Extract, Transform, Load) dos dados da camada Silver para a camada Gold. 
Focamos em manter apenas colunas relevantes para construção de dashboards futuros, como agregações temporais, geográficas e por severidade de incidentes.

## Colunas Mantidas para Dashboard
- timestamp: Para tendências temporais.
- orgid: Agregação por organização.
- detectorid: Detetores de alertas.
- alerttitle: Títulos de alertas.
- category: Categorias de incidentes.
- mitretechniques: Técnicas MITRE.
- incidentgrade: Severidade (target principal).
- entitytype: Tipos de entidades.
- evidencerole: Papel da evidência.
- osfamily: Família de SO.
- osversion: Versão de SO.
- lastverdict: Veredito final.
- countrycode, state, city: Localização geográfica.


## EXTRACT

Extraímos os dados do arquivo CSV da camada Silver.


In [29]:
import re
import os
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
from sqlalchemy import create_engine, text
import sqlalchemy

warnings.filterwarnings('ignore')

print("=== ETL Silver -> Gold (EXTRACT / TRANSFORM / LOAD) com DDL ===")

# ---------- Resolver caminho do repositório e DDL (preferência: data_layer/gold/ddl.sql) ----------
def resolve_repo_root():
    """
    Resolve a raiz do repositório de forma robusta:
    - Se estivermos dentro de uma pasta chamada 'transformer', sobe um nível.
    - Sobe a árvore de diretórios procurando por:
        1) <candidate>/data_layer/gold/ddl.sql  (preferido)
        2) <candidate>/ddl.sql
    - Retorna o primeiro diretório que contém um desses arquivos; caso nada seja encontrado,
      retorna o diretório atual (Path.cwd()).
    """
    cwd = Path.cwd()
    if cwd.name == 'transformer':
        cwd = cwd.parent

    for p in [cwd] + list(cwd.parents):
        if (p / 'data_layer' / 'gold' / 'ddl.sql').exists() or (p / 'ddl.sql').exists():
            return p
    return Path.cwd()

# resolver repo_root automaticamente
repo_root = resolve_repo_root()

# caminhos preferenciais
gold_ddl_path = repo_root / 'data_layer' / 'gold' / 'ddl.sql'
legacy_ddl_path = repo_root / 'ddl.sql'

# escolher ddl_path: preferir GOLD ddl, senão legacy, senão buscar recursivamente
if gold_ddl_path.exists():
    ddl_path = gold_ddl_path
elif legacy_ddl_path.exists():
    ddl_path = legacy_ddl_path
else:
    matches = list(repo_root.rglob('ddl.sql'))
    ddl_path = matches[0] if matches else None

# mensagens informativas
if ddl_path:
    print(f"Using DDL file at: {ddl_path}")
else:
    print("Warning: no ddl.sql found. Expected at '<repo_root>/data_layer/gold/ddl.sql' or '<repo_root>/ddl.sql'. "
          "Pass repo_root explicitly or add the DDL file to the repo.")

# ---------- Helpers ----------
def read_postgres_from_docker_compose(path: Path):
    """
    Lê credenciais básicas do docker-compose.yml (parsing simples).
    Retorna dict com chaves: user, password, db, host, port
    Se o arquivo não existir, retorna None.
    """
    if not path.exists():
        return None

    txt = path.read_text(encoding='utf-8')

    def find_var(k: str):
        pattern = rf'{re.escape(k)}\s*:\s*["\']?([^\n"\' ]+)'
        m = re.search(pattern, txt)
        return m.group(1).strip() if m else None

    user = find_var('POSTGRES_USER') or find_var('postgres_user') or 'postgres'
    password = find_var('POSTGRES_PASSWORD') or find_var('postgres_password') or 'postgres'
    db = find_var('POSTGRES_DB') or find_var('postgres_db') or find_var('POSTGRES_NAME') or 'microsoft-security'

    mport = re.search(r'ports\s*:\s*\n\s*-\s*["\']?(\d+):\d+', txt, re.MULTILINE)
    port = mport.group(1) if mport else None

    if not port:
        port = find_var('POSTGRES_PORT') or find_var('postgres_port')

    host = 'localhost'
    port = port or '5432'

    return dict(user=user, password=password, db=db, host=host, port=port)


def create_engine_from_compose(repo_root: Path):
    """
    Tenta criar uma engine SQLAlchemy lendo docker-compose.yml.
    Retorna (engine_or_None, pg_info_dict)
    """
    docker_path = repo_root / 'docker-compose.yml'
    pg = read_postgres_from_docker_compose(docker_path)
    if not pg:
        pg = {
            'user': os.getenv('POSTGRES_USER', 'postgres'),
            'password': os.getenv('POSTGRES_PASSWORD', 'postgres'),
            'db': os.getenv('POSTGRES_DB', 'microsoft-security'),
            'host': os.getenv('POSTGRES_HOST', 'localhost'),
            'port': os.getenv('POSTGRES_PORT', '5432')
        }

    url = f"postgresql+psycopg2://{pg['user']}:{pg['password']}@{pg['host']}:{pg['port']}/{pg['db']}"

    try:
        eng = create_engine(url)
        with eng.connect() as conn:
            pass
        print(f"Connected to Postgres: {pg['host']}:{pg['port']}/{pg['db']}")
        return eng, pg
    except Exception as e:
        print("Postgres unavailable via SQLAlchemy. Exception:", e)
        return None, pg


def execute_ddl(engine, ddl_path: Path):
    """
    Executa o arquivo DDL como um script único (não fragmenta em ';'),
    garantindo que blocos DO $$...$$ e CREATE TYPE sejam executados corretamente.
    Retorna True se rodou sem exceção.
    """
    if ddl_path is None or not ddl_path.exists():
        print(f"WARNING: DDL file not found at {ddl_path}")
        return False

    ddl_content = ddl_path.read_text(encoding='utf-8')
    print(f"Executing DDL from {ddl_path}...")

    try:
        # exec_driver_sql executa scripts que contenham múltiplas instruções
        with engine.begin() as conn:
            # SQLAlchemy connection offers exec_driver_sql which delegates to DBAPI
            conn.exec_driver_sql(ddl_content)
        print("DDL executed successfully. Schema 'dw' and tables created.")
        return True
    except AttributeError:
        # Fallback se exec_driver_sql não estiver disponível: usar raw_connection
        try:
            raw = engine.raw_connection()
            try:
                cur = raw.cursor()
                cur.execute(ddl_content)
                raw.commit()
                cur.close()
                raw.close()
                print("DDL executed successfully (fallback raw_connection).")
                return True
            except Exception as e2:
                raw.rollback()
                raw.close()
                print("Failed to execute DDL via raw_connection:", e2)
                return False
        except Exception as e3:
            print("Failed to get raw_connection for fallback DDL execution:", e3)
            return False
    except Exception as e:
        print(f"Failed to execute DDL: {e}")
        return False


def extract_silver(engine):
    """
    Lê os dados 'silver' diretamente do banco Postgres e aplica filtro
    para manter apenas os últimos 2 dias de dados (quando possível).
    """
    if engine is None:
        raise ConnectionError("No Postgres engine provided - cannot extract data.")

    df = None
    candidates = [
        "silver.microsoft_security_incident",
        "silver.microsoft_security",
        "silver.security_incident_prediction_silver",
        "microsoft_security_incident",
        "security_incident_prediction_silver"
    ]
    for t in candidates:
        try:
            print(f"Trying to read table: {t}")
            df = pd.read_sql_query(f"SELECT * FROM {t}", con=engine)
            print(f"Read {len(df)} rows from {t}")
            break
        except Exception as e:
            print(f" - couldn't read {t}: {str(e)}")
            continue

    if df is None:
        raise FileNotFoundError(f"No silver table found in Postgres. Tried: {candidates}")

    # --- Detectar coluna de timestamp (flexível) ---
    time_candidates = ['tsp', 'timestamp', 'created_at', 'time', 'event_time', 'ts']
    time_col = next((c for c in time_candidates if c in df.columns), None)

    if time_col:
        # converter para datetime (coerce para evitar erros)
        df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
        before_rows = len(df)
        before_mem_mb = df.memory_usage(deep=True).sum() / 1024**2

        # aplicar filtro: últimos 2 dias em relação ao max timestamp presente
        max_ts = df[time_col].max()
        if pd.isna(max_ts):
            print("Warning: timestamp column exists but contains only NaT after conversion. Skipping time filter.")
        else:
            cutoff = max_ts - pd.Timedelta(days=2)
            df = df[df[time_col] >= cutoff]
            after_rows = len(df)
            after_mem_mb = df.memory_usage(deep=True).sum() / 1024**2

            print(f"Applied 2-day filter on '{time_col}': {before_rows} -> {after_rows} rows")
            print(f"Estimated memory usage: {before_mem_mb:.2f} MB -> {after_mem_mb:.2f} MB")
    else:
        print("No obvious timestamp column found (tried: {}). Skipping 2-day filter.".format(", ".join(time_candidates)))

    # Garantir account_sid como string para joins posteriores (se existir)
    if 'account_sid' in df.columns:
        df['account_sid'] = df['account_sid'].astype(str)

    print("EXTRACT complete. Shape:", df.shape)
    return df


=== ETL Silver -> Gold (EXTRACT / TRANSFORM / LOAD) com DDL ===
Using DDL file at: c:\Users\fabio\OneDrive\Área de Trabalho\SBD2-2025-2\data_layer\gold\ddl.sql


## TRANSFORM

Realizamos transformações: seleção de colunas relevantes, conversão de tipos e criação das tabelas dimensionais e fato.


In [30]:
# ---------- TRANSFORM ----------
def transform_to_gold(df: pd.DataFrame):
    """
    Transformações para 'gold' seguindo o modelo dimensional do DDL:
    - Cria tabelas dimensionais (Dim_tmp, Dim_org, Dim_sis)
    - Cria tabela fato (Fat_inc)
    
    Modificações importantes:
    * Detecta automaticamente a coluna de tempo entre várias candidatas.
    * Evita merges de grande custo: usa .map() para associar srk_org e srk_sis.
    * Garante tipos (account_sid/device_id -> str) antes do map.
    """
    print("Starting TRANSFORM...")
    df = df.copy()

    # Normaliza nomes de colunas (snake_case)
    df.columns = [str(c).strip().lower().replace(' ', '_') for c in df.columns]

    # Remove duplicatas
    before_dup = len(df)
    df.drop_duplicates(inplace=True)
    after_dup = len(df)
    print(f"Dropped {before_dup - after_dup} duplicate rows.")

    # Detectar coluna de tempo entre candidatas
    time_candidates = ['timestamp', 'tsp', 'created_at', 'time', 'event_time', 'ts']
    time_col = next((c for c in time_candidates if c in df.columns), None)
    if time_col:
        df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
        print(f"Using time column: '{time_col}' (parsed to datetime).")
    else:
        print("No obvious timestamp column found (tried: {}). Time dimension will be empty.".format(", ".join(time_candidates)))

    # ===== DIMENSÃO TEMPO (Dim_tmp) =====
    print("Creating Dim_tmp (Time Dimension)...")
    dim_tmp = pd.DataFrame()
    if time_col:
        # criar strings padronizadas para fazer mapping (YYYY-MM-DD HH:MM:SS)
        tsp_series = df[time_col].dt.strftime('%Y-%m-%d %H:%M:%S')
        unique_ts = pd.Index(tsp_series.dropna().unique())
        dim_tmp = pd.DataFrame({'tsp': unique_ts})
        dim_tmp['srk_tmp'] = range(1, len(dim_tmp) + 1)
        dim_tmp = dim_tmp[['srk_tmp', 'tsp']]
    print(f"Dim_tmp created with {len(dim_tmp)} records")

    # ===== DIMENSÃO ORGANIZAÇÃO (Dim_org) =====
    print("Creating Dim_org (Organization Dimension)...")
    org_cols = ['account_sid', 'account_upn', 'countrycode', 'state', 'city']
    org_cols_present = [c for c in org_cols if c in df.columns]

    if org_cols_present:
        dim_org = df[org_cols_present].drop_duplicates().reset_index(drop=True)
        dim_org['srk_org'] = range(1, len(dim_org) + 1)

        # Renomeia para match com DDL
        rename_map = {
            'account_sid': 'acc_sid',
            'account_upn': 'acc_upn',
            'countrycode': 'ctr_cod',
            'state': 'sta',
            'city': 'cty'
        }
        dim_org.rename(columns={k: v for k, v in rename_map.items() if k in dim_org.columns}, inplace=True)

        # Garante todas as colunas do DDL
        for col in ['acc_sid', 'acc_upn', 'ctr_cod', 'sta', 'cty']:
            if col not in dim_org.columns:
                dim_org[col] = None

        dim_org = dim_org[['srk_org', 'acc_sid', 'acc_upn', 'ctr_cod', 'sta', 'cty']]
    else:
        dim_org = pd.DataFrame(columns=['srk_org', 'acc_sid', 'acc_upn', 'ctr_cod', 'sta', 'cty'])

    print(f"Dim_org created with {len(dim_org)} records")

    # ===== DIMENSÃO SISTEMA/ENTIDADE (Dim_sis) =====
    print("Creating Dim_sis (System/Entity Dimension)...")
    sis_cols = ['device_id', 'sha', 'url', 'osfamily', 'osversion', 'detector_id', 'alert_title', 'entity_type']
    sis_cols_present = [c for c in sis_cols if c in df.columns]

    if sis_cols_present:
        dim_sis = df[sis_cols_present].drop_duplicates().reset_index(drop=True)
        dim_sis['srk_sis'] = range(1, len(dim_sis) + 1)

        # Renomeia para match com DDL
        rename_map = {
            'device_id': 'dev_id',
            'osfamily': 'osf',
            'osversion': 'osv',
            'detector_id': 'dtc_id',
            'alert_title': 'alt_ttl',
            'entity_type': 'ent_typ'
        }
        dim_sis.rename(columns={k: v for k, v in rename_map.items() if k in dim_sis.columns}, inplace=True)

        # Garante todas as colunas do DDL
        for col in ['dev_id', 'sha', 'url', 'osf', 'osv', 'dtc_id', 'alt_ttl', 'ent_typ']:
            if col not in dim_sis.columns:
                dim_sis[col] = None

        dim_sis = dim_sis[['srk_sis', 'dev_id', 'sha', 'url', 'osf', 'osv', 'dtc_id', 'alt_ttl', 'ent_typ']]
    else:
        dim_sis = pd.DataFrame(columns=['srk_sis', 'dev_id', 'sha', 'url', 'osf', 'osv', 'dtc_id', 'alt_ttl', 'ent_typ'])

    print(f"Dim_sis created with {len(dim_sis)} records")

     # ===== TABELA FATO (Fat_inc) =====
    print("Creating Fat_inc (Fact Table)...")
    fat_inc = df.copy()

    # 1) ASSOCIAR srk_tmp (usando map — dim_tmp é pequena geralmente)
    if not dim_tmp.empty and time_col in fat_inc.columns:
        # garantir mesma forma de chave
        fat_inc['tsp'] = fat_inc[time_col].dt.strftime('%Y-%m-%d %H:%M:%S')
        tmp_map = dim_tmp.set_index('tsp')['srk_tmp']
        fat_inc['srk_tmp'] = fat_inc['tsp'].map(tmp_map)
        fat_inc.drop(columns=['tsp'], inplace=True)
    else:
        fat_inc['srk_tmp'] = None

    # 2) ASSOCIAR srk_org via map (muito mais leve que merge)
    if not dim_org.empty and 'account_sid' in fat_inc.columns:
        # preparar mapa: garantir strings e unicidade
        dim_org['acc_sid'] = dim_org['acc_sid'].astype(str)
        temp_org = dim_org[['srk_org', 'acc_sid']].drop_duplicates(subset='acc_sid').set_index('acc_sid')['srk_org']
        # garantir account_sid como string no fato
        fat_inc['account_sid'] = fat_inc['account_sid'].astype(str)
        fat_inc['srk_org'] = fat_inc['account_sid'].map(temp_org)
    else:
        fat_inc['srk_org'] = None

    # 3) ASSOCIAR srk_sis via map (device_id)
    if not dim_sis.empty and 'device_id' in fat_inc.columns:
        dim_sis['dev_id'] = dim_sis['dev_id'].astype(str)
        temp_sis = dim_sis[['srk_sis', 'dev_id']].drop_duplicates(subset='dev_id').set_index('dev_id')['srk_sis']
        fat_inc['device_id'] = fat_inc['device_id'].astype(str)
        fat_inc['srk_sis'] = fat_inc['device_id'].map(temp_sis)
    else:
        fat_inc['srk_sis'] = None

    # Estatísticas rápidas sobre correspondências (útil para debug)
    if 'srk_org' in fat_inc.columns:
        matched_org = fat_inc['srk_org'].notna().sum()
        print(f"srk_org matches: {matched_org} / {len(fat_inc)} ({matched_org/len(fat_inc):.2%})")
    if 'srk_sis' in fat_inc.columns:
        matched_sis = fat_inc['srk_sis'].notna().sum()
        print(f"srk_sis matches: {matched_sis} / {len(fat_inc)} ({matched_sis/len(fat_inc):.2%})")

    # Renomeia colunas do fato conforme seu mapa original
    fact_cols_map = {
        'id': 'srk_inc',
        'alert_id': 'alt_id',
        'category': 'cat',
        'mitre_techniques': 'mtr_tcn',
        'incident_grade': 'inc_gde',
        'evidence_role': 'evd_rol',
        'last_verdict': 'lst_vrd'
    }

    for old_col, new_col in fact_cols_map.items():
        if old_col in fat_inc.columns:
            fat_inc.rename(columns={old_col: new_col}, inplace=True)

    # Seleciona colunas finais da tabela fato (mantém a ordem desejada quando presentes)
    fact_final_cols = ['srk_inc', 'srk_tmp', 'srk_org', 'srk_sis', 'alt_id', 'cat', 'mtr_tcn', 'inc_gde', 'evd_rol', 'lst_vrd']
    fat_inc = fat_inc[[c for c in fact_final_cols if c in fat_inc.columns]]

    # CRIAR srk_inc ÚNICO: se houver duplicatas, usar row_number
    if 'srk_inc' in fat_inc.columns:
        # Verificar duplicatas
        duplicates = fat_inc['srk_inc'].duplicated().sum()
        if duplicates > 0:
            print(f"Warning: Found {duplicates} duplicate srk_inc values. Generating unique row IDs...")
            fat_inc['srk_inc'] = range(1, len(fat_inc) + 1)
        else:
            # Se srk_inc é único, garantir que seja int64
            fat_inc['srk_inc'] = fat_inc['srk_inc'].astype('int64')
    else:
        # Se srk_inc não existe, gerar sequencial
        fat_inc['srk_inc'] = range(1, len(fat_inc) + 1)

    for col in ['srk_tmp', 'srk_org', 'srk_sis']:
        if col in fat_inc.columns:
            fat_inc[col] = fat_inc[col].astype('Int64')

    print(f"Fat_inc created with {len(fat_inc)} records")
    print("TRANSFORM complete.")

    return {
        'dim_tmp': dim_tmp,
        'dim_org': dim_org,
        'dim_sis': dim_sis,
        'fat_inc': fat_inc
    }

## LOAD

Carregamos os dados processados para a camada Gold (schema 'dw').


In [31]:
def load_gold_to_postgres(tables_dict: dict, engine):
    """
    Grava as tabelas dimensionais e fato no schema 'dw' do Postgres.
    
    Melhorias:
    - Diferencia TRUNCATE entre tabelas SERIAL (com RESTART IDENTITY) e BIGINT PRIMARY KEY
    - Valida dados nulos em chaves primárias
    - Usa append após truncate (mais seguro)
    - Fallback robusto para inserções que falham
    """
    if engine is None:
        raise ConnectionError("No Postgres engine provided - cannot load data.")

    schema = 'dw'

    try:
        inspector = sqlalchemy.inspect(engine)

        # garante schema
        with engine.begin() as conn:
            conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {schema}"))

        # Ordem de carga: dimensões primeiro, depois fato
        # (key, table_name, is_bigint_pk)
        load_order = [
            ('dim_tmp', 'dim_tmp', False),
            ('dim_org', 'dim_org', False),
            ('dim_sis', 'dim_sis', False),
            ('fat_inc', 'fat_inc', True)  # fat_inc tem BIGINT PRIMARY KEY
        ]

        for key, table_name, is_bigint_pk in load_order:
            if key not in tables_dict:
                print(f"⚠ Skipping {table_name} (not found in tables_dict)")
                continue

            df = tables_dict[key]
            if df is None or df.empty:
                print(f"⚠ Skipping {table_name} (empty DataFrame)")
                continue

            tbl = table_name.lower()
            print(f"Loading {tbl} ({len(df)} records) into schema '{schema}'...")

            # VALIDAÇÃO: verificar NULLs em colunas PRIMARY KEY
            if is_bigint_pk and 'srk_inc' in df.columns:
                nulls_in_pk = df['srk_inc'].isna().sum()
                if nulls_in_pk > 0:
                    print(f" ✗ Error: {nulls_in_pk} NULL values in srk_inc (PRIMARY KEY). Cannot insert.")
                    return False
                # Converter srk_inc para int64 (seguro, sem NULLs)
                df['srk_inc'] = df['srk_inc'].astype('int64')

            # verificar se a tabela existe
            exists = inspector.has_table(tbl, schema=schema)

            with engine.begin() as conn:
                if exists:
                    # Truncate diferenciado por tipo de chave primária
                    try:
                        if is_bigint_pk:
                            # fat_inc: BIGINT PRIMARY KEY, sem RESTART IDENTITY
                            conn.execute(text(f"TRUNCATE TABLE {schema}.{tbl} CASCADE"))
                        else:
                            # dimensões: SERIAL, com RESTART IDENTITY
                            conn.execute(text(f"TRUNCATE TABLE {schema}.{tbl} RESTART IDENTITY CASCADE"))
                        print(f" - Truncated existing table {schema}.{tbl}")
                    except Exception as e:
                        print(f" - Warning truncating table {schema}.{tbl}: {e}")

            # Tentar gravar com to_sql
            try:
                if exists:
                    # Após truncate, usar 'append' (tabela vazia e estruturada)
                    df.to_sql(
                        tbl,
                        con=engine,
                        schema=schema,
                        if_exists='append',
                        index=False,
                        method='multi',
                        chunksize=5000
                    )
                else:
                    # Primeira vez: 'replace' cria a tabela
                    df.to_sql(
                        tbl,
                        con=engine,
                        schema=schema,
                        if_exists='replace',
                        index=False,
                        method='multi',
                        chunksize=5000
                    )
                print(f" ✓ {schema}.{tbl} loaded successfully")
                
            except Exception as e:
                # Fallback: inserir em chunks menores, sem method='multi'
                print(f" - Primary insert failed: {e}")
                print(f" - Attempting fallback (chunked insert without multi)...")
                try:
                    batch_size = 500
                    n = len(df)
                    for start in range(0, n, batch_size):
                        end = min(start + batch_size, n)
                        df.iloc[start:end].to_sql(
                            tbl,
                            con=engine,
                            schema=schema,
                            if_exists='append' if (exists or start > 0) else 'replace',
                            index=False,
                            method=None
                        )
                    print(f" ✓ {schema}.{tbl} loaded successfully (fallback)")
                    
                except Exception as e2:
                    print(f" ✗ Fallback also failed for {schema}.{tbl}: {e2}")
                    return False

        print(f"\n✓ All tables loaded to schema '{schema}' successfully!")
        return True

    except Exception as e:
        print(f"✗ Failed to write to Postgres (load phase): {e}")
        return False

## Execução do ETL Completo


In [32]:
# ---------- Main ETL flow ----------
def run_etl():
    """
    Fluxo principal do ETL:
    1. Conecta ao Postgres
    2. Executa o DDL para criar schema e tabelas
    3. Extrai dados da camada Silver
    4. Transforma em modelo dimensional (dimensões + fato)
    5. Carrega no schema 'dw'
    """
    print("="*60)
    print("STARTING ETL PROCESS")
    print("="*60)
    
    # 1. Conectar ao Postgres
    engine, pg = create_engine_from_compose(repo_root)
    if engine is None:
        print("Cannot proceed: Postgres engine unavailable. Exiting ETL.")
        return
    
    # 2. Executar DDL
    print("\n" + "="*60)
    print("STEP 1: Executing DDL")
    print("="*60)
    if not execute_ddl(engine, ddl_path):
        print("Warning: DDL execution failed or file not found. Proceeding anyway...")
    
    # 3. Extract
    print("\n" + "="*60)
    print("STEP 2: EXTRACT - Reading from Silver layer")
    print("="*60)
    try:
        raw = extract_silver(engine)
    except Exception as e:
        print("EXTRACT failed:", e)
        return
    
    # 4. Transform
    print("\n" + "="*60)
    print("STEP 3: TRANSFORM - Creating dimensional model")
    print("="*60)
    tables_dict = transform_to_gold(raw)
    
    # 5. Load
    print("\n" + "="*60)
    print("STEP 4: LOAD - Writing to Gold layer (schema 'dw')")
    print("="*60)
    written = load_gold_to_postgres(tables_dict, engine)
    
    if written:
        print("\n" + "="*60)
        print(" ETL COMPLETED SUCCESSFULLY!")
        print("="*60)
        print(f"\nDimensional model created in schema 'dw':")
        print(f"  - Dim_tmp: {len(tables_dict['dim_tmp'])} records")
        print(f"  - Dim_org: {len(tables_dict['dim_org'])} records")
        print(f"  - Dim_sis: {len(tables_dict['dim_sis'])} records")
        print(f"  - Fat_inc: {len(tables_dict['fat_inc'])} records")
    else:
        print("\n ETL failed during LOAD phase.")


# Executar ETL
if __name__ == '__main__':
    run_etl()

STARTING ETL PROCESS
Connected to Postgres: localhost:5433/microsoft-security

STEP 1: Executing DDL
Executing DDL from c:\Users\fabio\OneDrive\Área de Trabalho\SBD2-2025-2\data_layer\gold\ddl.sql...
DDL executed successfully. Schema 'dw' and tables created.

STEP 2: EXTRACT - Reading from Silver layer
Trying to read table: silver.microsoft_security_incident
Read 9516837 rows from silver.microsoft_security_incident
Applied 2-day filter on 'timestamp': 9516837 -> 242708 rows
Estimated memory usage: 8057.45 MB -> 207.31 MB
EXTRACT complete. Shape: (242708, 23)

STEP 3: TRANSFORM - Creating dimensional model
Starting TRANSFORM...
Dropped 34231 duplicate rows.
Using time column: 'timestamp' (parsed to datetime).
Creating Dim_tmp (Time Dimension)...
Dim_tmp created with 32737 records
Creating Dim_org (Organization Dimension)...
Dim_org created with 25993 records
Creating Dim_sis (System/Entity Dimension)...
Dim_sis created with 23902 records
Creating Fat_inc (Fact Table)...
srk_org matches: